# Notebook: Golden Gate cloning with recipe

# Assembly Reports Workflow — Notebook Guide

This guide explains what the notebook does, how to configure it, and how each code block fits together. It is tailored to the **Plasmidio** in‑silico pipeline for generating **Golden Gate/Gibson/3‑G** assembly reports and extracting final **GenBank** records.

---

## 1) Purpose

- **Load part libraries** (Promoters, RBS, Genes, Terminators, Backbones) from your project folders.  
- **Read TU designs** from an Excel template (`GG_designs.xlsx`, sheet `TUs`).  
- **Generate assembly reports** (`*_report.zip`) for the specified designs.  
- **Extract the final constructs** from each report into a single folder for downstream analysis.  
- **Optionally clean up features** by removing near‑duplicates (tolerance ±3 bp).

The flow is: **Folders → Designs (Excel) → Reports (ZIP) → Final .gb/.gbk → Feature clean‑up**.

---

## 2) Prerequisites

- Python environment with your **Plasmidio/assembly‑designer** package installed (for `generate_recipe_assemblies`, `organize_assembly_reports`, `remove_near_duplicate_features`).  
- A project directory that contains your part folders (see below).  
- An Excel file `GG_designs.xlsx` with a sheet named **`TUs`** that lists your designs.

Suggested environment steps (example):
```bash
conda create -n adesigner python=3.11 -y
conda activate adesigner
pip install uv
uv pip install -e ".[insilico]"
```

---

## 3) Folder structure & configuration

### Base directory
```python
BASE_DIR = Path.cwd()
```
- The **base directory** of your project; all part folders are expected **inside** this directory.  
- If your notebook sits somewhere else, set an **absolute** path, e.g.:
  ```python
  BASE_DIR = Path(r"C:\Repo\assembly_designer_test")
  ```

### Part folders (category → path)
```python
folder_paths = [
    BASE_DIR / "Promoter_parts",
    BASE_DIR / "RBS_parts",
    BASE_DIR / "Gene_of_interest_parts",
    BASE_DIR / "Terminator_parts",
    BASE_DIR / "Backbone_parts",
]
```
- Keep the order consistent with `category_order` (next section).  
- You **don’t** need to change your existing folder layout; we just reference it.

### Report output
```python
report_folder = BASE_DIR
```
- Target folder for generated reports and extracted GenBank files.  
- You can route outputs to a subfolder: `report_folder = BASE_DIR / "reports"`.

### Optional gene filter
```python
gene_filter = "None"  # or "" / None to include all
```
- Placeholder for narrowing designs to specific genes (by filename). Not applied by default in the shown snippet.

### Category order (naming & layout)
```python
category_order = ["Promoter", "RBS", "Gene", "Terminator", "Backbone"]
category_dirs = dict(zip(category_order, folder_paths))
```
- Defines the **logical order** in assemblies and how construct names are assembled.  
- `category_dirs` maps **category name → folder path** and is passed into the report generator.

---

## 4) Safety check — validate folders

```python
existing_paths = [str(p) for p in folder_paths if Path(p).exists()]
if not existing_paths:
    raise FileNotFoundError("No valid parts folders found — please check the paths.")
```
- Ensures your part folders exist before long‑running steps start.  
- If the check fails, verify `BASE_DIR` and folder names.

---

## 5) Load TU designs from Excel

```python
designs = pd.read_excel("GG_designs.xlsx", sheet_name="TUs").to_dict("records")
designs
```
- Reads the **`TUs`** sheet and converts it to a list of dicts that the pipeline expects.  
- Make sure the sheet and column names match what your recipes/functions require.

---

## 6) Generate reports for selected recipes

```python
reports = generate_recipe_assemblies(
    category_dirs=category_dirs,
    designs=designs,
    category_order=category_order,
    output_dir=report_folder / "reports",
)
```
- Builds assembly **report ZIPs** (`*_report.zip`) for the specified `designs`.  
- Uses your category mapping and order to resolve parts and naming.  
- Output goes to `<report_folder>/reports`.

**Tip:** If you want to run only a subset (e.g., filtered genes), either prefilter `designs` or extend the function to accept a filter.

---

## 7) Extract final constructs (GenBank)

```python
gb_paths = organize_assembly_reports(
    report_dir=report_folder / "reports",
    reports=reports,
    delete_zip=False,
    final_only=True,  # ensures exactly one final per ZIP
)
print("Number of final constructs:", len(gb_paths))
print("Constructs:", [p.name for p in gb_paths])
```
- Extracts the **final construct** from each ZIP into `<reports>/Assembly`.  
- Keeps the ZIP files (`delete_zip=False`) so you can re‑inspect the full report.  
- `final_only=True` guarantees a single `.gb/.gbk` per report.

---

## 8) Feature clean‑up (recommended)

```python
assembly_folder = Path("reports") / "Assembly"

for file_name in os.listdir(assembly_folder):
    if file_name.endswith(".gb") or file_name.endswith(".gbk"):
        file_path = os.path.join(assembly_folder, file_name)
        remove_near_duplicate_features(file_path, tolerance=3)  # ±3 bp
```
- Removes **near‑duplicate features** (e.g., overlapping or repeated annotations).  
- The `tolerance` controls how close features may be to be considered duplicates.

> Run this step if you observe over‑annotated constructs or near‑identical features.

---

## 9) Tips & troubleshooting

- **Paths not found**: Check `BASE_DIR` and folder names (typos, capitalization).  
- **Excel not found or wrong sheet**: Confirm `GG_designs.xlsx` is next to the notebook or provide an absolute path; verify the `TUs` sheet name.  
- **No final outputs**: Inspect the generated ZIPs in `<reports>`; errors may be summarized inside.  
- **Feature clutter**: Keep `remove_near_duplicate_features` enabled; adjust `tolerance` if needed.  
- **Custom naming/order**: Update `category_order` to match the way you want constructs named and assembled.

---

## 10) Minimal end‑to‑end checklist

1. Set `BASE_DIR` to your project root.  
2. Ensure the five part folders exist inside `BASE_DIR`.  
3. Place `GG_designs.xlsx` (with `TUs` sheet) next to the notebook or adjust the path.  
4. Run the **reports** cell — expect ZIPs in `<BASE_DIR>/reports`.  
5. Run the **extract finals** cell — expect `.gb/.gbk` in `<BASE_DIR>/reports/Assembly`.  
6. (Optional) Run **feature clean‑up** to tidy annotations.  
7. Use the final GenBank files for downstream analysis/visualization.


In [1]:
# ------------------------------ Imports & setup

from __future__ import annotations

from pathlib import Path
import os  # optional; keep if you use e.g. os.environ / os.listdir
import pandas as pd

# Project API (top-level re-exports)
from assembly_designer import (
    # Reports / recipes
    generate_recipe_assemblies,
    organize_assembly_reports,
)

# Project API (submodule-only: not re-exported at top level)
from assembly_designer.plasmidio import (
    load_dna_file,
    remove_near_duplicate_features,
    _safe_filename,
)

# Optional dependency sanity check:
# - dnacauldron: required to simulate assemblies & write *_report.zip
# - snapgene_reader: enables reading .dna (SnapGene) files
try:
    import dnacauldron  # noqa: F401
    from snapgene_reader import snapgene_file_to_dict  # noqa: F401
    print("Optional deps OK: dnacauldron, snapgene-reader")
except Exception as err:
    print("⚠️ Optional deps check:", err)



c:\Users\stoltmann\AppData\Local\mambaforge\envs\adesigner\Lib\site-packages\Bio\pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


Optional deps OK: dnacauldron, snapgene-reader


## 1. Golden Gate Assembly

In [2]:
# ------------------------------ Base directory for your project. All part folders are expected inside this dir.
# Tip: If your notebook is not located next to the part folders, replace Path.cwd()
# with an absolute path, e.g. Path(r"C:\Repo\assembly_designer_test").
BASE_DIR = Path.cwd()

# Paths to the category folders (keep the order consistent with category_order below).
# You DO NOT need to change your folder structure; we just reference the existing ones.
folder_paths = [
    BASE_DIR / "Promoter_parts",
    BASE_DIR / "RBS_parts",
    BASE_DIR / "Gene_of_interest_parts",
    BASE_DIR / "Terminator_parts",
    BASE_DIR / "Backbone_parts",
]

# Target folder for all generated reports (ZIPs) and extracted GenBank files.
# You can point this to a subfolder if you prefer, e.g. BASE_DIR / "reports".
report_folder = BASE_DIR

# Optional: a string to filter which gene(s) to include (by filename).
# Note: The combinatorial function shown earlier does not use this directly.
# If you want this filter applied, either rename files or extend the function
# to accept a filter. For now, it’s just a placeholder variable.
gene_filter = "None"  # or "" / None to include all

In [3]:
# ------------------------------ Define the category order exactly as you want parts laid out in each assembly.
# This order also controls how assembly names are constructed.
category_order = ["Promoter", "RBS", "Gene", "Terminator", "Backbone"]

# Convert your existing list of folders to a mapping: Category -> Folder path.
# The order of 'folder_paths' must match 'category_order' above.
category_dirs = dict(zip(category_order, folder_paths))

In [4]:
# ------------------------------ Filter out non-existent folders (robust)
existing_paths = [str(p) for p in folder_paths if Path(p).exists()]
if not existing_paths:
    raise FileNotFoundError("No valid parts folders found — please check the paths.")

In [5]:
# ------------------------------ import designs from the Excel template 
designs = pd.read_excel("GG_designs.xlsx", sheet_name="TUs").to_dict("records")
designs

[{'Constructs': 'C1',
  'TU': 'TU1',
  'Promoter': 'J23100_AB',
  'RBS': 'B0032m_BC',
  'Gene': 'DVA_ecPanD',
  'Terminator': 'B0015_DF',
  'Backbone': 'DVK_AF'},
 {'Constructs': 'C2',
  'TU': 'TU2',
  'Promoter': 'J23102_AB',
  'RBS': 'B0032m_BC',
  'Gene': 'DVA_ecPanD',
  'Terminator': 'B0015_DF',
  'Backbone': 'DVK_AF'},
 {'Constructs': 'C3',
  'TU': 'TU3',
  'Promoter': 'J23107_AB',
  'RBS': 'B0033m_BC',
  'Gene': 'DVA_ecPanD',
  'Terminator': 'B0015_DF',
  'Backbone': 'DVK_AF'},
 {'Constructs': 'C4',
  'TU': 'TU4',
  'Promoter': 'J23116_AB',
  'RBS': 'B0034m_BC',
  'Gene': 'DVA_ecPanD',
  'Terminator': 'B0015_DF',
  'Backbone': 'DVK_AF'},
 {'Constructs': 'C5',
  'TU': 'TU5',
  'Promoter': 'J23100_AB',
  'RBS': 'BCD2_BC',
  'Gene': 'DVA_ecPanD',
  'Terminator': 'B0015_DF',
  'Backbone': 'DVK_AF'},
 {'Constructs': 'C6',
  'TU': 'TU6',
  'Promoter': 'J23102_AB',
  'RBS': 'BCD2_BC',
  'Gene': 'DVA_ecPanD',
  'Terminator': 'B0015_DF',
  'Backbone': 'DVK_AF'},
 {'Constructs': 'C7',
  'T

In [6]:
# ------------------------------ Build reports for exactly these two recipes
reports = generate_recipe_assemblies(
    category_dirs=category_dirs,          # your mapping: {category: folder}
    designs=designs,
    category_order=category_order,
    output_dir=report_folder / "reports",
)

# Extract only the final construct from each ZIP into <reports>/Assembly
gb_paths = organize_assembly_reports(
    report_dir=report_folder / "reports",
    reports=reports,
    delete_zip=False,
    final_only=True,                      # <- ensures 1 file per ZIP
)

print("Number of final constructs:", len(gb_paths))
print("Constructs:", [p.name for p in gb_paths])

Number of final constructs: 8
Constructs: ['J23100_AB_B0032m_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb', 'J23102_AB_B0032m_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb', 'J23107_AB_B0033m_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb', 'J23116_AB_B0034m_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb', 'J23100_AB_BCD2_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb', 'J23102_AB_BCD2_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb', 'J23107_AB_BCD8_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb', 'J23116_AB_BCD12_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb']


In [ ]:
# ------------------------------ delete near-duplicate features in all extracted GenBank files
# (optional, but recommended if your parts have overlapping annotations)
assembly_folder = Path("reports") / "Assembly"

for file_name in os.listdir(assembly_folder):
    if file_name.endswith(".gb") or file_name.endswith(".gbk"):
        file_path = os.path.join(assembly_folder, file_name)
        remove_near_duplicate_features(file_path, tolerance=3)  # Tolerance of ±3 bases

✅ Cleaned file saved: reports\Assembly\J23100_AB_B0032m_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb
✅ Cleaned file saved: reports\Assembly\J23100_AB_BCD2_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb
✅ Cleaned file saved: reports\Assembly\J23102_AB_B0032m_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb
✅ Cleaned file saved: reports\Assembly\J23102_AB_BCD2_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb
✅ Cleaned file saved: reports\Assembly\J23107_AB_B0033m_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb
✅ Cleaned file saved: reports\Assembly\J23107_AB_BCD8_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb
✅ Cleaned file saved: reports\Assembly\J23116_AB_B0034m_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb
✅ Cleaned file saved: reports\Assembly\J23116_AB_BCD12_BC_DVA_ecPanD_B0015_DF_DVK_AF.gb
